<div style="
    font-family: 'Trebuchet MS'; 
    padding: 30px; 
    border-radius: 60px; 
    background: linear-gradient(135deg, rgba(33,87,62,1), rgb(92,152,255));
    color: white;
    text-align: center;
    box-shadow: 0 8px 22px rgba(0,0,0,0.25);
">
    <h1 style="font-family: Trebuchet MS; padding: 12px; font-size: 48px; color:rgba(33, 87, 62, 1); text-align: center; line-height: 1.25;">
    <b>⚽Post Match<span style="color: #000000"> Notebook 🎮📉</span></b><br>
  <span style="color: #000000; font-size: 24px">features contained :</span><br>
  <span style="color: #000000; font-size: 18px">✨ provide the exhuastion status for each player ✨</span><br>
  <span style="color: #000000; font-size: 18px">✨ provide training for each player and the cluster of positions (pre_match redundant feature) ✨</span>
</h1>

</div>

In [15]:
api_base = r'https://football-backend-app.victoriouswater-69fff737.swedencentral.azurecontainerapps.io/'

In [16]:
import numpy as np , json , requests
import pandas as pd
from google import genai
from pandas import DataFrame , Series
from datetime import datetime
import os

# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">1. | Fatigue & Injury Risk Predictor </div>

In [17]:
# i'll be using the `get_team_lnm` function from the pre_match to spedup the process of getting the data
def get_team_lnm(api_base :str , team_id:int , num_matchs: int) -> dict :
  '''this function takes the base of the api without the endpoint , the team id and the number of last matchs wanted
  and returns a dictionary of the last matchs ids of the team with the home team and away team names'''
  try :
    response = requests.get(api_base +f'teams/{team_id}/events/last/0') # api response
  except:
    return 'the api is down'
  match_info = { # extracting match details
    match['id']: {
        'homeTeam': match['homeTeam']['name'],
        'awayTeam': match['awayTeam']['name']
    }
    for match in response.json()['events']
}
  match_info =  dict(reversed(list(match_info.items())[-num_matchs:])) # filtering the last n matches from the data and reverse them (the last match is the first)
  match_info['target_team_id'] =team_id  # adding the id of the team to the dict
  match_info['target_team_name'] = requests.get(api_base + f'teams/{team_id}').json( ).get('team').get('name') # adding the name of the team to the dict
  return match_info

In [18]:
def get_fatigue(api_base :str , team_id: int) -> dict: 
    '''this function is used to get the fatigue index and injury risk level of the players of a team based on the minutes played in the last match '''
    players_details = [] # list to add players details
    matches_info = get_team_lnm(api_base , team_id , 1)  # getting the last match info of the team
 
    for match_id in matches_info: # getting the id for the last match and checking if it's an int (not the team id or name)
        if isinstance(match_id, int):
            is_home = matches_info.get('target_team_name') == matches_info.get(match_id).get('homeTeam') # getting if the target team is home or away
            team_key = 'home' if is_home else 'away'
            
            try:
                data = requests.get(api_base + f'events/{match_id}/lineups').json() # getting the lineups data for the match
                
                
                if team_key in data and isinstance(data[team_key], dict) and 'players' in data[team_key]: # accessing only our players
                    players = data[team_key]['players']
                    
                    for player in players:
                        if not isinstance(player, dict): continue

                       # getting only requiered data
                        players_details.append(
                            {
                                'player_id': player.get('player').get('id'), 
                                'name' :  player.get('player').get('name'),
                                'position' :  player.get('position') ,
                                'minutes_played' : player.get('statistics').get('minutesPlayed', 0) ,
                            }
                        )
            except Exception as e:
                print(f"Error fetching lineups for match {match_id}: {e}")
                return  
    
    players_df = pd.DataFrame(players_details)
    # getting fatigue index as percentage
    min_s = players_df['minutes_played'].min()
    max_s = players_df['minutes_played'].max()
    players_df['fatigue_index'] = round(100 * (players_df['minutes_played'] - min_s) / (max_s - min_s) if max_s != min_s else 0)
    
    # constructing injury risk level column
    players_df['injury_risk_level'] = players_df.apply(
        lambda row: 'High' if row['minutes_played'] >= 180 else ('Low' if row['fatigue_index'] < 60 else ('Moderate' if row['fatigue_index'] < 80 else 'High')), 
        axis=1
    )

    # constructing this json part
    players_analysis = [
        {
            "player_id": str(row["player_id"]),
            "name": row["name"],
            "position": row["position"],
            "minutes_played": int(row["minutes_played"]),
            "fatigue_and_risk": {
                "fatigue_index": float(row["fatigue_index"]),
                "injury_risk_level": row["injury_risk_level"]
            }
        }
        for _, row in players_df.iterrows()
    ]

    result = {"players_analysis": players_analysis}
    return result 
data = get_fatigue(api_base , 2829)
data

{'players_analysis': [{'player_id': '857574',
   'name': 'Andriy Lunin',
   'position': 'G',
   'minutes_played': 90,
   'fatigue_and_risk': {'fatigue_index': 100.0, 'injury_risk_level': 'High'}},
  {'player_id': '795064',
   'name': 'Trent Alexander-Arnold',
   'position': 'D',
   'minutes_played': 89,
   'fatigue_and_risk': {'fatigue_index': 99.0, 'injury_risk_level': 'High'}},
  {'player_id': '822519',
   'name': 'Éder Militão',
   'position': 'D',
   'minutes_played': 90,
   'fatigue_and_risk': {'fatigue_index': 100.0, 'injury_risk_level': 'High'}},
  {'player_id': '142622',
   'name': 'Antonio Rudiger',
   'position': 'D',
   'minutes_played': 90,
   'fatigue_and_risk': {'fatigue_index': 100.0, 'injury_risk_level': 'High'}},
  {'player_id': '792073',
   'name': 'Ferland Mendy',
   'position': 'D',
   'minutes_played': 90,
   'fatigue_and_risk': {'fatigue_index': 100.0, 'injury_risk_level': 'High'}},
  {'player_id': '835485',
   'name': 'Brahim Díaz',
   'position': 'M',
   'minute

# <div style="font-family: Trebuchet MS; background-color:rgba(33, 87, 62, 1); color: #FFFFFF; padding: 12px; line-height: 1.5;">2. | training </div>

In [24]:
from dotenv import load_dotenv

load_dotenv(override=True) 
api_key = os.environ.get("GEMINI_API_KEY_POST_MATCH")
client = genai.Client(api_key=api_key) 
def get_training_recommendations(api_base: str, team_id: int) -> str:
    """
    Fetches the stats of the SINGLE most recent match and sends them to the LLM to identify weaknesses and suggest training exercises.
    Forces output to tightly match the final_json schema for 'trainingPlan'.
    """
    try:
        matches_info = get_team_lnm(api_base, team_id, 1)
        match_id = None
        for k in matches_info.keys():
            if isinstance(k, int): 
                match_id = k
                break
                
        if not match_id:
            return '{"error": "No recent matches found"}'
            
        is_home = matches_info.get('target_team_name') == matches_info.get(match_id).get('homeTeam')
        team_key = 'home' if is_home else 'away'
        
        stats_data = []
        try:
            lineup_resp = requests.get(api_base + f'events/{match_id}/lineups').json()
            if team_key in lineup_resp and 'players' in lineup_resp[team_key]:
                for p in lineup_resp[team_key]['players']:
                    stats = p.get('statistics', {})
                    mins = stats.get('minutesPlayed', 0)
                    if mins > 0: 
                        clean_stats = {k:v for k,v in stats.items() if v} 
                        stats_data.append({
                            "playerId": p.get('player', {}).get('id'),
                            "playerName": p.get('player', {}).get('name'),
                            "stats": clean_stats
                        })
        except Exception as e:
            pass

        if not stats_data:
            return '{"error": "No player statistics found for the last match"}'
        
        
        prompt = f"""
        Analyze the following player statistics from their SINGLE MOST RECENT MATCH and identify weaknesses.
        Generate a strictly valid JSON training plan. DO NOT use markdown code blocks (e.g., no ```json).
        The JSON MUST strictly follow this exact structure and key names:
        {{
          "trainingPlan": {{
            "teamDrills": [
              {{
                "focusCode": "str (e.g., PRESS_RESISTANCE)",
                "priority": "str (HIGH/MEDIUM/LOW)",
                "linkedOpponentFeature": "str (e.g., HIGH_PRESS_INTENSITY)",
                "targetedPositions": ["D", "M"] 
              }}
            ],
            "individualDrills": [
              {{
                "playerId": int,
                "playerName": "str",
                "drillCode": "str (e.g., 1V1_DEFENDING_WIDE)"
              }}
            ]
          }}
        }}

        Player Statistics:
        {json.dumps(stats_data)}
        """

        
        response = client.models.generate_content(
            model = "gemini-2.5-flash",
            config={
                "system_instruction": "You are a professional football tactician. Output only strictly valid, unformatted JSON that perfectly matches the requested schema. No explanations."
            },
            contents=prompt
        )
        return response.text.replace("```json", "").replace("```", "").strip()

    except Exception as e:
        return f'{{"error": "LLM API failed: {e}"}}'

# Execution Block:
try:
    print("Fetching advanced training recommendations from LLM for the LAST match...\n")
    training_result_json = get_training_recommendations(api_base, 2829)
    
    # Validate and print JSON
    parsed_json = json.loads(training_result_json)
    print(json.dumps(parsed_json, indent=2))
    
    # Save to file properly avoiding markdown truncation 
    with open('training_plan_output.json', 'w', encoding='utf-8') as f:
        json.dump(parsed_json, f, indent=2)
        
except json.JSONDecodeError:
    print("Error parsing JSON. The LLM might have returned invalid formatting:\n")
    print(training_result_json)
except Exception as e:
    print(f"Execution Error: {e}")


Fetching advanced training recommendations from LLM for the LAST match...

{
  "trainingPlan": {
    "teamDrills": [
      {
        "focusCode": "BUILD_UP_PLAY_UNDER_PRESS",
        "priority": "HIGH",
        "linkedOpponentFeature": "HIGH_PRESS_INTENSITY",
        "targetedPositions": [
          "G",
          "D",
          "M"
        ]
      },
      {
        "focusCode": "DEFENSIVE_SHAPE_TRANSITION",
        "priority": "HIGH",
        "linkedOpponentFeature": "QUICK_ATTACKS_TRANSITIONS",
        "targetedPositions": [
          "D",
          "M"
        ]
      },
      {
        "focusCode": "MIDFIELD_PROGRESSION_RETENTION",
        "priority": "MEDIUM",
        "linkedOpponentFeature": "COMPACT_MIDFIELD_BLOCK",
        "targetedPositions": [
          "M",
          "F"
        ]
      }
    ],
    "individualDrills": [
      {
        "playerId": 857574,
        "playerName": "Andriy Lunin",
        "drillCode": "GK_DISTRIBUTION_LONG"
      },
      {
        "playerId": 